In [ ]:
from pathlib import Path
from pytubefix import YouTube
from pydub import AudioSegment
from pydub.utils import which

def download_youtube_audio(
    url: str,
    destination: str | Path,
    start_ms: int | None = None,
    end_ms: int | None = None,
    output_filename: str | None = None,
    bitrate: str = "192k",
) -> Path:
    # --- Ensure ffmpeg/ffprobe are available (adjust if needed) ---
    ffmpeg_path  = which("ffmpeg")  or r"C:\ffmpeg\bin\ffmpeg.exe"
    ffprobe_path = which("ffprobe") or r"C:\ffmpeg\bin\ffprobe.exe"
    AudioSegment.converter = ffmpeg_path
    AudioSegment.ffprobe   = ffprobe_path

    dest = Path(destination)
    dest.mkdir(parents=True, exist_ok=True)

    # 1) Download audio-only stream (container often .webm or .m4a)
    yt = YouTube(url)
    stream = yt.streams.filter(only_audio=True).first()
    downloaded_path = Path(stream.download(output_path=dest))

    # 2) Load without forcing a format; ffmpeg will detect it
    audio = AudioSegment.from_file(downloaded_path)

    # 3) Resolve trim bounds (ms)
    if start_ms is None: start_ms = 0
    if end_ms   is None: end_ms   = len(audio)

    # Clamp to valid range
    start_ms = max(0, start_ms)
    end_ms   = min(len(audio), end_ms)
    if start_ms >= end_ms:
        raise ValueError(f"Invalid trim window: start_ms={start_ms}, end_ms={end_ms}, length={len(audio)}")

    segment = audio[start_ms:end_ms]

    # 4) Output filename/path
    if not output_filename:
        # default name: original stem + trimmed seconds
        output_filename = f"{downloaded_path.stem}_{start_ms//1000}-{end_ms//1000}.mp3"
    out_path = dest / output_filename  # <-- file path, not directory

    # 5) Export to mp3
    segment.export(out_path, format="mp3", bitrate=bitrate)
    return out_path


In [36]:
out = download_youtube_audio(
    url="https://www.youtube.com/watch?v=BdX1XeSxeKQ&t=4863s",
    destination=r"E:\Github\Block A 2\Y 2 Block A\extract audio",
    start_ms=20 * 60 * 1000,
    end_ms=40 * 60 * 1000,
    output_filename="متری شیش و نیم_20to40.mp3"
)
print("Saved to:", out)


Saved to: E:\Github\Block A 2\Y 2 Block A\extract audio\متری شیش و نیم_20to40.mp3


In [38]:
out = download_youtube_audio(
    url="https://www.youtube.com/watch?v=8yX8KFa2NFo&t=1875s",
    destination=r"E:\Github\Block A 2\Y 2 Block A\extract audio",
    start_ms=27 * 60 * 1000,
    end_ms=47 * 60 * 1000,
    output_filename="همرفیق_20to40.mp3"
)
print("Saved to:", out)

Saved to: E:\Github\Block A 2\Y 2 Block A\extract audio\همرفیق_20to40.mp3


In [8]:
import re
import assemblyai as aai
import pandas as pd
from pathlib import Path

def assemblyai_to_csv(path: str | Path, api_key: str, csv_path: str | Path | None = None) -> str:
    # 1) Transcribe
    aai.settings.api_key = api_key
    audio_path = Path(path)
    transcript = aai.Transcriber().transcribe(
        str(audio_path),
        config=aai.TranscriptionConfig(
            language_code="fa",
            punctuate=True,
            format_text=True,
            speaker_labels=True,   # enables utterances (preferred)
            boost_param="high",
        )
    )
    if transcript.error:
        raise RuntimeError(f"AssemblyAI error: {transcript.error}")

    # 2) Build rows (utterances preferred)
    rows = []
    if getattr(transcript, "utterances", None):
        for i, u in enumerate(transcript.utterances, start=1):
            rows.append({"line": i, "text": (u.text or "").strip()})
    else:
        # Fallback: split the full text into sentences using Persian-friendly punctuation
        full_text = (transcript.text or "").strip()
        # Split on a boundary where a sentence-ending punctuation is followed by whitespace
        sentences = [s.strip() for s in re.split(r'(?<=[\.!\?؟…])\s+', full_text) if s.strip()]
        for i, s in enumerate(sentences, start=1):
            rows.append({"line": i, "text": s})

    # 3) Save CSV (UTF-8 with BOM so Persian renders correctly in Excel on Windows)
    df = pd.DataFrame(rows, columns=["line", "text"])
    out_csv = Path(csv_path) if csv_path else audio_path.with_suffix(".csv")
    df.to_csv(out_csv, index=False, encoding="utf-8-sig")
    return str(out_csv)


In [10]:
metri_6= assemblyai_to_csv(path=r"E:\Github\Block A 2\Y 2 Block A\extract audio\متری شیش و نیم_20to40.mp3",
               api_key="536ece37a704451c9b4e0b1872eadf70",
               csv_path=r"E:\Github\Block A 2\Y 2 Block A\extract audio\CSV\Assemblyai\metri_6.csv")
print("Transcript saved to:", metri_6)

ham_refigh= assemblyai_to_csv(path=r"E:\Github\Block A 2\Y 2 Block A\extract audio\همرفیق_20to40.mp3",
               api_key="536ece37a704451c9b4e0b1872eadf70",
               csv_path=r"E:\Github\Block A 2\Y 2 Block A\extract audio\CSV\Assemblyai\ham_refigh.csv")

print("Transcript saved to:", ham_refigh)


Transcript saved to: E:\Github\Block A 2\Y 2 Block A\extract audio\CSV\Assemblyai\metri_6.csv
Transcript saved to: E:\Github\Block A 2\Y 2 Block A\extract audio\CSV\Assemblyai\ham_refigh.csv


In [11]:
import csv
import whisper
from pathlib import Path

def whisper_to_csv(media_path: str | Path, model_name: str = "large-v3", csv_path: str | None = None) -> str:
    media_path = Path(media_path)
    if not media_path.exists():
        raise FileNotFoundError(f"File not found: {media_path}")

    # Load model and transcribe (Persian; deterministic)
    model = whisper.load_model(model_name)
    result = model.transcribe(
        str(media_path),
        language="fa",
        temperature=0.0,
        beam_size=5,
        best_of=1,
        fp16=False  # set True if you have a CUDA GPU
    )

    # Build rows: one row per segment; fallback to whole text if no segments
    rows = []
    segments = result.get("segments") or []
    if segments:
        for i, seg in enumerate(segments, start=1):
            rows.append({"line": i, "text": (seg.get("text") or "").strip()})
    else:
        rows.append({"line": 1, "text": (result.get("text") or "").strip()})

    # Output CSV path (UTF-8 BOM so Persian shows correctly in Excel)
    out_csv = Path(csv_path) if csv_path else media_path.with_suffix(".csv")
    with open(out_csv, "w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["line", "text"])
        writer.writeheader()
        writer.writerows(rows)

    return str(out_csv)


In [12]:

csv_file = whisper_to_csv(
    r"E:\Github\Block A 2\Y 2 Block A\extract audio\متری شیش و نیم_20to40.mp3",
    csv_path=r"E:\Github\Block A 2\Y 2 Block A\extract audio\CSV\Whisper\metri_6.csv"  # choose a folder you can write to
)
print("Saved CSV:", csv_file)

csv_file = whisper_to_csv(
    r"E:\Github\Block A 2\Y 2 Block A\extract audio\همرفیق_20to40.mp3",
    csv_path=r"E:\Github\Block A 2\Y 2 Block A\extract audio\CSV\Whisper\ham_refigh.csv"# choose a folder you can write to
)
print("Saved CSV:", csv_file)


Saved CSV: E:\Github\Block A 2\Y 2 Block A\extract audio\CSV\Whisper\metri_6.csv
Saved CSV: E:\Github\Block A 2\Y 2 Block A\extract audio\CSV\Whisper\ham_refigh.csv
